# 7.2 · Corpus atlas: PCA, t-SNE, and the original space

**Question.** Which Marvel pages are close as documents, and how faithfully does a two-dimensional map show those relationships?

This notebook completes [exercise 7.2, “Build a corpus atlas”](../data/week7/Week%207%20%C2%B7%20From%20meaning%20to%20measurement%20%C2%B7%2002805.html). It embeds all 303 raw pages, inspects cosine neighbors in the original space, projects the **same** vectors with PCA and t-SNE, and colors both maps with communities from the earlier link network. Run the cells from top to bottom. The final chart needs a live Jupyter widget frontend for hover and click.


## Setup and data

Install these packages in the **same environment as the notebook kernel** if needed:

```bash
python -m pip install numpy scikit-learn sentence-transformers plotly ipywidgets anywidget
```

The first model run downloads `sentence-transformers/all-MiniLM-L6-v2` weights. The page archive is already in `notebooks/data/marvel_pages.zip`; `data/graph.json` holds the structural community labels. The path cell accepts a kernel started either in `notebooks/` or at the repository root.


In [1]:
from pathlib import Path
from urllib.parse import unquote
from zipfile import ZipFile
import json

from IPython.display import display, HTML
import numpy as np

root = Path.cwd()
if not (root / "notebooks" / "data" / "marvel_pages.zip").exists():
    root = root.parent

archive_path = root / "notebooks" / "data" / "marvel_pages.zip"
graph_path = root / "data" / "graph.json"
with ZipFile(archive_path) as archive:
    page_files = sorted(
        name for name in archive.namelist()
        if name.startswith("marvel_pages/") and name.endswith(".txt")
        and not name.endswith("README.txt")
    )
    pages = {
        unquote(Path(name).stem): archive.read(name).decode("utf-8-sig")
        for name in page_files
    }

graph = json.loads(graph_path.read_text(encoding="utf-8"))
node_by_id = {node["id"]: node for node in graph["nodes"]}
communities = {group["id"]: group for group in graph["communities"]}
names = sorted(pages)
texts = [pages[name] for name in names]
index = {name: i for i, name in enumerate(names)}
assert len(names) == 303 and set(names) == set(node_by_id)
print(f"{len(names)} raw pages · {len(communities)} network communities")
print(f"Archive: {archive_path}")


303 raw pages · 8 network communities
Archive: c:\Users\victo\Codex\Go_Nuts_repos\notebooks\data\marvel_pages.zip


## 1 · Embed complete pages through passages

`all-MiniLM-L6-v2` produces one 384-dimensional vector per input, but its input limit is much shorter than most of these Wikipedia pages. Passing whole pages directly would silently emphasize only their opening text. Here, we split each page at sentence boundaries into passages of **at most 180 whitespace-separated words**. A sentence longer than the cap is split into word blocks. Any passage still over the model's token limit is split again at a word boundary until it fits. We embed every passage, average its passage vectors, and normalize the page mean. The cell reports the model's actual token limit and the longest final passage in tokenizer units.

The resulting `embeddings` array is the source of truth for cosine neighbors and both maps. Neither projection writes back into it.


In [2]:
import re
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_ID)

def passages(text, max_words=180):
    result, current = [], []
    for sentence in re.split(r"(?<=[.!?])\s+", text.strip()):
        words = sentence.split()
        while words:
            room = max_words - len(current)
            current.extend(words[:room])
            words = words[room:]
            if len(current) == max_words:
                result.append(" ".join(current))
                current = []
    if current:
        result.append(" ".join(current))
    return result

def fit_token_limit(passage):
    token_count = len(model.tokenizer(passage, add_special_tokens=True, truncation=False)["input_ids"])
    if token_count <= model.max_seq_length:
        return [passage]
    words = passage.split()
    if len(words) == 1:
        raise ValueError("A single word exceeds the model token limit; inspect the source page.")
    middle = len(words) // 2
    return fit_token_limit(" ".join(words[:middle])) + fit_token_limit(" ".join(words[middle:]))

page_passages = [
    [part for passage in passages(text) for part in fit_token_limit(passage)]
    for text in texts
]
all_passages = [passage for group in page_passages for passage in group]
lengths = np.array([len(group) for group in page_passages])
token_lengths = model.tokenizer(
    all_passages, add_special_tokens=True, truncation=False
)["input_ids"]
over_limit = sum(len(ids) > model.max_seq_length for ids in token_lengths)
print(f"Model: {MODEL_ID}")
print(f"Model maximum input: {model.max_seq_length} tokens")
print(f"Pages: {len(names)} · passages: {len(all_passages)} · median passages/page: {np.median(lengths):.0f}")
print(f"Longest passage: {max(map(len, token_lengths))} tokens · over model limit: {over_limit}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (293 > 256). Running this sequence through the model will result in indexing errors


Model: sentence-transformers/all-MiniLM-L6-v2
Model maximum input: 256 tokens
Pages: 303 · passages: 4732 · median passages/page: 8
Longest passage: 256 tokens · over model limit: 0


**Token-limit check.** The final over-limit count should be zero. The extra splits preserve text when “180 words” exceeds the model's token budget. The assertion below makes the check explicit before encoding begins. Progress output stays visible because the corpus takes some time.


In [3]:
assert over_limit == 0, "Reduce the passage word cap until every passage fits the model."
passage_vectors = model.encode(
    all_passages, batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)
embeddings = np.empty((len(names), passage_vectors.shape[1]), dtype=np.float32)
start = 0
for i, count in enumerate(lengths):
    embeddings[i] = passage_vectors[start:start + count].mean(axis=0)
    start += count
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
assert embeddings.shape == (303, 384)
print("Page embedding matrix:", embeddings.shape)


Batches:   0%|          | 0/148 [00:00<?, ?it/s]

Page embedding matrix: (303, 384)


## 2 · Nearest documents in 384 dimensions

We inspect **Storm**, **Doctor Strange**, and **Spider-Man**. For normalized vectors, a dot product is cosine similarity. The five neighbors below come from the original 384-dimensional page vectors, not from either plot. A high score describes similarity under this encoder and chunking rule; it does not establish friendship, team membership, or a link in the network.


In [4]:
from html import escape

subjects = ["Storm_(Marvel_Comics)", "Doctor_Strange", "Spider-Man"]
similarity = embeddings @ embeddings.T
np.fill_diagonal(similarity, -np.inf)

def label(name):
    return node_by_id[name]["name"]

def cosine_neighbors(name, k=5):
    row = similarity[index[name]]
    closest = np.argsort(row)[::-1][:k]
    return [(names[j], float(row[j])) for j in closest]

for subject in subjects:
    print(f"\n{label(subject)} — five closest pages by original-space cosine")
    for rank, (neighbor, score) in enumerate(cosine_neighbors(subject), 1):
        print(f"  {rank}. {label(neighbor):38s} {score:.3f}")



Storm (Marvel Comics) — five closest pages by original-space cosine
  1. Emma Frost                             0.710
  2. Surge (Marvel Comics)                  0.701
  3. Jean Grey                              0.699
  4. Cyclops (Marvel Comics)                0.669
  5. Jubilee (character)                    0.662

Doctor Strange — five closest pages by original-space cosine
  1. Doctor Druid                           0.810
  2. Jericho Drumm                          0.688
  3. Miracleman (character)                 0.632
  4. Alice Nugent                           0.625
  5. Captain Universe                       0.594

Spider-Man — five closest pages by original-space cosine
  1. Spider-Man Noir                        0.896
  2. Spider-Man (Marvel Mangaverse)         0.872
  3. Mayday Parker                          0.827
  4. Scarlet Spider                         0.826
  5. Spider-Girl                            0.826


### Read the pages behind the scores

The next cell shows each subject's opening passage, followed by every neighbor's opening passage and, when present, a short context around the subject's name in the neighbor's page. Read these excerpts before interpreting the scores. To inspect a complete article, run `print(pages["page_id"])` with the page ID from the output. The excerpts make it possible to check whether a pair is close because of shared teams, powers, genre, recurring names, or simply the style of the article.


In [5]:
def excerpt(name, words=115):
    return " ".join(pages[name].split()[:words]) + " …"

def mention_context(page_id, subject_id, radius=250):
    text = pages[page_id]
    term = label(subject_id).split(" (")[0]
    match = re.search(re.escape(term), text, flags=re.IGNORECASE)
    if not match:
        return "No direct name mention found in this page."
    return "…" + text[max(0, match.start()-radius):match.end()+radius].replace("\n", " ") + "…"

for subject in subjects:
    blocks = [f"<h3>{escape(label(subject))}</h3>",
              f"<p><b>Subject page:</b> {escape(excerpt(subject))}</p>"]
    for neighbor, score in cosine_neighbors(subject):
        blocks.append(
            f"<details><summary>{escape(label(neighbor))} · cosine {score:.3f}</summary>"
            f"<p><b>Opening:</b> {escape(excerpt(neighbor))}</p>"
            f"<p><b>Subject mention:</b> {escape(mention_context(neighbor, subject))}</p>"
            "</details>"
        )
    display(HTML("".join(blocks)))


**Interpretation to test against the displayed pages.** Storm's close pages should largely reflect X-Men and mutant stories, while Doctor Strange's should reflect magic and mysticism. Spider-Man's neighbors may mix his supporting cast, variants, and fellow street-level heroes. The actual ranked lists above decide which of these explanations is supported; a direct name mention is evidence of shared story context, while two similar openings without a mention suggest shared subject matter. The excerpt cell deliberately exposes the text so these claims can be checked rather than inferred from a scatterplot.


## 3 · Two projections of the unchanged vectors

PCA finds two directions that retain as much linear variance as possible. t-SNE optimizes a view of local neighborhoods and its axes have no direct semantic meaning. The same `embeddings` array feeds both methods. A t-SNE gap between islands is not a calibrated cosine distance, and a dot close in either map may be far down the original-space neighbor list.


In [6]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

pca = PCA(n_components=2, random_state=42)
pca_xy = pca.fit_transform(embeddings)
tsne_xy = TSNE(
    n_components=2, perplexity=30, init="pca",
    learning_rate="auto", random_state=42,
).fit_transform(embeddings)
print(f"PCA variance retained in two axes: {pca.explained_variance_ratio_.sum():.1%}")
print("Map shapes:", pca_xy.shape, tsne_xy.shape)
assert embeddings.shape == (303, 384)


PCA variance retained in two axes: 10.0%
Map shapes: (303, 2) (303, 2)


## 4 · Where the picture disagrees with cosine

For each chosen character, the table compares the five closest **dots** on each 2D map with the five closest **documents** in 384D. The overlap is a simple local check, not a score for the entire map. A particularly misleading case is selected automatically: among the three subjects, it finds a dot that looks like a nearest 2D neighbor but ranks poorly by original-space cosine. The code then prints a concrete sentence naming the pair and shows their source excerpts. This sentence is computed from the model run, so it does not claim a specific mismatch in advance.


In [7]:
def visual_neighbors(xy, subject, k=5):
    i = index[subject]
    distances = np.linalg.norm(xy - xy[i], axis=1)
    distances[i] = np.inf
    return np.argsort(distances)[:k]

def original_rank(subject, candidate):
    return int(np.flatnonzero(np.argsort(similarity[index[subject]])[::-1] == index[candidate])[0] + 1)

misleading = []
for subject in subjects:
    true_five = {name for name, _ in cosine_neighbors(subject)}
    print(f"\n{label(subject)} — original-space top 5: {', '.join(map(label, true_five))}")
    for method, xy in [("PCA", pca_xy), ("t-SNE", tsne_xy)]:
        visual = [names[j] for j in visual_neighbors(xy, subject)]
        overlap = len(true_five.intersection(visual))
        print(f"  {method:5s} overlap {overlap}/5 · closest dots: {', '.join(map(label, visual))}")
        for position, candidate in enumerate(visual, 1):
            misleading.append((original_rank(subject, candidate), position, method, subject, candidate))

rank, dot_position, method, subject, candidate = max(misleading)
actual_first, actual_score = cosine_neighbors(subject, 1)[0]
print(f"\nConcrete mismatch: On the {method} map, {label(candidate)} is dot #{dot_position} "
      f"nearest to {label(subject)}, but ranks #{rank} by cosine in 384D "
      f"(similarity {similarity[index[subject], index[candidate]]:.3f}). "
      f"The true cosine-nearest page is {label(actual_first)} ({actual_score:.3f}).")
display(HTML(
    f"<h3>Read this mismatch</h3>"
    f"<p><b>{escape(label(subject))}:</b> {escape(excerpt(subject))}</p>"
    f"<p><b>Visually close · {escape(label(candidate))}:</b> {escape(excerpt(candidate))}</p>"
    f"<p><b>Cosine closest · {escape(label(actual_first))}:</b> {escape(excerpt(actual_first))}</p>"
))



Storm (Marvel Comics) — original-space top 5: Jubilee (character), Jean Grey, Emma Frost, Surge (Marvel Comics), Cyclops (Marvel Comics)
  PCA   overlap 0/5 · closest dots: Anole (character), Phyla-Vell, Blackthorn (character), Korvus, Thena
  t-SNE overlap 3/5 · closest dots: Jean Grey (X-Men film series), Surge (Marvel Comics), Jubilee (character), Anole (character), Jean Grey

Doctor Strange — original-space top 5: Alice Nugent, Jericho Drumm, Doctor Druid, Miracleman (character), Captain Universe
  PCA   overlap 1/5 · closest dots: Black Panther (character), Captain Universe, Stingray (Marvel Comics), Iron Man (Ultimate Marvel character), Blue Marvel
  t-SNE overlap 2/5 · closest dots: Doctor Druid, Jericho Drumm, Paladin (comics), Straw Man (comics), Man-Thing

Spider-Man — original-space top 5: Spider-Man (Marvel Mangaverse), Spider-Man Noir, Mayday Parker, Scarlet Spider, Spider-Girl
  PCA   overlap 3/5 · closest dots: Spider-Man Noir, Scarlet Spider, Spider-Man (Marvel Mangave

**How to explain the mismatch.** The printout gives the exact pair, its 2D position among neighbors, its original-space rank, and the genuine cosine-nearest page. Read the three excerpts: shared teams, powers, character names, and story settings can explain high-dimensional similarity. The projection retains only two coordinates, so many distinctions between 384D vectors collapse; t-SNE additionally rearranges global spacing to emphasize local neighborhoods. The rank disagreement is the evidence that the plotted closeness is misleading for this pair. The excerpts tell you what content was compressed away, rather than the 2D map itself.


## 5 · Interactive corpus atlas, colored by link community

Each point is a page. Color comes from the **network's structural community** in `data/graph.json`, not from the text encoder. Hover for the page name and community. Click any dot to list its five cosine-nearest documents from the **original 384D vectors**. PCA and t-SNE use the coordinates already computed above; coloring and clicking do not recompute or change them. The community legend can help spot where text similarity and link structure agree or diverge.


In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets

fig = go.FigureWidget(make_subplots(
    rows=1, cols=2, subplot_titles=("PCA · variance view", "t-SNE · neighborhood view"),
    horizontal_spacing=0.08,
))

for column, xy in [(1, pca_xy), (2, tsne_xy)]:
    for community_id, group in communities.items():
        idx = [i for i, name in enumerate(names)
               if node_by_id[name]["comm"] == community_id]
        fig.add_trace(go.Scattergl(
            x=xy[idx, 0], y=xy[idx, 1], mode="markers",
            name=group["label"], legendgroup=str(community_id),
            showlegend=(column == 1),
            text=[label(names[i]) for i in idx],
            customdata=idx,
            hovertemplate="<b>%{text}</b><br>" + group["label"] + "<extra></extra>",
            marker=dict(color=group["color"], size=9, opacity=0.83,
                        line=dict(color="white", width=0.6)),
        ), row=1, col=column)

fig.update_layout(
    template="plotly_white", height=630, width=1150,
    paper_bgcolor="#f8f7f3", plot_bgcolor="#f8f7f3",
    font=dict(family="Arial, sans-serif", color="#26323b", size=13),
    margin=dict(l=45, r=20, t=85, b=90),
    legend=dict(orientation="h", y=-0.14, x=0, font=dict(size=11)),
    hoverlabel=dict(bgcolor="white", font_size=13),
)
fig.update_xaxes(title_text="Component 1", showgrid=True, gridcolor="#e8e7e2", zeroline=False)
fig.update_yaxes(title_text="Component 2", showgrid=True, gridcolor="#e8e7e2", zeroline=False)

details = widgets.Output(layout={"border": "1px solid #ddd", "padding": "12px"})

def clicked(trace, points, selector):
    if not points.point_inds:
        return
    page_id = names[int(trace.customdata[points.point_inds[0]])]
    rows = "".join(
        f"<li>{escape(label(other))} <span style='color:#52616b'>· cosine {score:.3f}</span></li>"
        for other, score in cosine_neighbors(page_id)
    )
    with details:
        details.clear_output(wait=True)
        display(HTML(
            f"<h3 style='margin:0 0 6px'>{escape(label(page_id))}</h3>"
            f"<p style='margin:0 0 8px'>Five nearest documents in the original 384D space:</p>"
            f"<ol style='margin:0'>{rows}</ol>"
        ))

for trace in fig.data:
    trace.on_click(clicked)

display(fig, details)


FigureWidget({
    'data': [{'customdata': [6, 15, 31, 36, 42, 50, 58, 71, 80, 81, 85, 96, 98,
                             104, 105, 108, 116, 140, 141, 146, 147, 154, 156, 157,
                             161, 183, 187, 192, 196, 199, 203, 213, 216, 217, 230,
                             260, 263, 286, 287, 294, 297, 298, 299],
              'hovertemplate': '<b>%{text}</b><br>X-Men<extra></extra>',
              'legendgroup': '0',
              'marker': {'color': '#4e79a7', 'line': {'color': 'white', 'width': 0.6}, 'opacity': 0.83, 'size': 9},
              'mode': 'markers',
              'name': 'X-Men',
              'showlegend': True,
              'text': [Anole (character), Betsy Braddock, Blue Bullet, Box
                       (comics), Cable (character), Ch'od, Cyclops (Marvel Comics),
                       Doctor Nemesis, Emma Frost, Empath (character), Fabio
                       Medina, G. W. Bridge, Garrison Kane, Gladiator (Kallark),
                       Golem 

Output(layout=Layout(border_bottom='1px solid #ddd', border_left='1px solid #ddd', border_right='1px solid #dd…

### Reading the atlas

- **Start with a click, then check the text.** The panel reports cosine neighbors in 384D even if other dots look closer on screen.
- **Compare PCA with t-SNE.** Look for a subject whose apparent nearby dots change between the panels; the underlying cosine list stays fixed.
- **Compare color with location.** A mixed-color region means documents from different link communities have similar text representations there. A same-color region suggests agreement, but color alone is no proof of semantic similarity.
- **Limits.** Passage averaging loses order across passages; article length and boilerplate affect the mean. PCA discards most dimensions, and t-SNE can distort global distances. Re-run with a different chunk rule or seed only if you are deliberately testing those modeling choices.
